In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json

SYMPTOM_WEIGHTS = {
    'cough_weeks':  0.25,
    'weight_loss':  0.20,
    'night_sweats': 0.15,
    'fever':        0.15,
    'hiv_positive': 0.15,
    'tb_contact':   0.10
}

def symptom_risk_score(symptoms):
    return round(sum(SYMPTOM_WEIGHTS[k] * symptoms.get(k, 0)
                     for k in SYMPTOM_WEIGHTS), 4)

def combined_prediction(image_prob, symptom_score):
    combined = 0.7 * image_prob + 0.3 * symptom_score
    result   = 'REFER FOR SPUTUM TEST' if combined >= 0.5 else 'LOW RISK'
    return round(combined, 4), result

# four patient scenarios
scenarios = [
    {'label': 'High risk patient',   'symptoms': {'cough_weeks':1,'weight_loss':1,'night_sweats':1,'fever':1,'hiv_positive':1,'tb_contact':1}, 'img_prob': 0.95},
    {'label': 'Medium risk patient', 'symptoms': {'cough_weeks':1,'weight_loss':1,'night_sweats':0,'fever':0,'hiv_positive':0,'tb_contact':1}, 'img_prob': 0.72},
    {'label': 'Low risk patient',    'symptoms': {'cough_weeks':1,'weight_loss':0,'night_sweats':0,'fever':0,'hiv_positive':0,'tb_contact':0}, 'img_prob': 0.30},
    {'label': 'No symptoms',         'symptoms': {'cough_weeks':0,'weight_loss':0,'night_sweats':0,'fever':0,'hiv_positive':0,'tb_contact':0}, 'img_prob': 0.10},
]

print(f"{'Patient':<25} {'Symptoms':>10} {'Image':>7} {'Combined':>10} {'Decision'}")
print("-" * 70)
for s in scenarios:
    sym   = symptom_risk_score(s['symptoms'])
    comb, result = combined_prediction(s['img_prob'], sym)
    print(f"{s['label']:<25} {sym:>10.4f} {s['img_prob']:>7.2f} {comb:>10.4f} {result}")

# save
with open('/content/drive/MyDrive/EdgeTB_Project/models/symptom_module.json', 'w') as f:
    json.dump({'weights': SYMPTOM_WEIGHTS, 'image_weight': 0.7,
               'symptom_weight': 0.3, 'threshold': 0.5}, f, indent=2)
print("\nSymptom module saved to Drive")

Patient                     Symptoms   Image   Combined Decision
----------------------------------------------------------------------
High risk patient             1.0000    0.95     0.9650 REFER FOR SPUTUM TEST
Medium risk patient           0.5500    0.72     0.6690 REFER FOR SPUTUM TEST
Low risk patient              0.2500    0.30     0.2850 LOW RISK
No symptoms                   0.0000    0.10     0.0700 LOW RISK

Symptom module saved to Drive


In [1]:
import tensorflow as tf, tf_keras
print(tf.__version__, tf_keras.__version__)

2.21.0 2.21.0
